# Лаборатория 10 — Риск-менеджмент в виде кода

Профессиональная обёртка, доведённая до конкретики. Вы построите:

1. **Калькулятор размера позиции**, превращающий `analyzer.max_loss` в число акций/спредов по правилу
   фиксированного % риска.
2. **Агрегатор греков портфеля** по трём открытым позициям на DEMO с проверкой против лимитов.
3. **Чек-лист в виде кода** — предторговую функцию, которая отклоняет сделку, провалившую любое
   правило.

Работает офлайн, сверху вниз, на цепочке DEMO (спот 100, IV ~25%).

In [ ]:
import math
from optionslab import strategies, analyzer, greeks
r = lambda x: round(float(x), 2)
SPOT, VOL = 100.0, 0.26

## 1. Размер позиции по максимальному убытку

Рискуйте фиксированным небольшим % капитала в каждой сделке, отмеряя размер по **максимальному
убытку** позиции (а не по премии): `units = floor(equity * risk% / |max_loss|)`. Округление **вниз**.

In [ ]:
def size_position(pos, equity, risk_pct):
    ml = analyzer.max_loss(pos)
    if ml == float("-inf") or ml == 0:
        return None, ml                       # неограниченный риск: размер по стресс-тесту, а не по этой формуле
    dollars_at_risk = equity * risk_pct
    units = math.floor(dollars_at_risk / abs(ml))
    return units, ml

In [ ]:
ic = strategies.iron_condor((87.5,0.37),(92.5,1.01),(107.5,1.19),(112.5,0.43), expiry=45/365)
for eq in (25_000, 50_000):
    units, ml = size_position(ic, eq, 0.02)
    print(f"капитал ${eq}: 2% = ${eq*0.02:.0f} под риском, max_loss кондора ${r(ml)} -> {units} кондор(ов)")

На \$25k при 2% вы торгуете **один** кондор (`$500 / $360 = 1.38`, округление вниз).
Обратите внимание, что функция делает с **неограниченным риском**: голый короткий пут возвращает
`-inf`, поэтому формула отказывается считать размер — неограниченный риск надо мерить
стресс-сценарием либо не брать вовсе.

In [ ]:
naked = strategies.short_put((95, 1.58), expiry=45/365)
print("голый короткий пут — units, max_loss:", size_position(naked, 25_000, 0.02))

## 2. Агрегация греков портфеля

Три открытые позиции. Сложите их долларовые греки (`greeks.position_greeks` складывается), чтобы
увидеть реальную экспозицию *книги* — а она часто не такая, какой её рисует любая отдельная сделка.

In [ ]:
bps = strategies.bull_put_spread((95, 1.58), (90, 0.62), expiry=45/365)
bcs = strategies.bull_call_spread((100, 3.91), (110, 0.73), expiry=45/365)
book = [ic, bps, bcs]
total = greeks.Greeks(0, 0, 0, 0, 0)
for p in book:
    g = greeks.position_greeks(p, SPOT, VOL)
    print(f"{p.label[:30]:30} d={r(g.delta):>7} t={r(g.theta):>6} v={r(g.vega):>6}")
    total = total + g
print(f"{'ПОРТФЕЛЬ':30} d={r(total.delta):>7} t={r(total.theta):>6} v={r(total.vega):>6}")

Эта книга в **чистом лонге по дельте (+51)** и в **чистом шорте по веге (−8)**: несмотря на
один «нейтральный» кондор, направление задают два бычьих спреда, а вегу — короткий по волатильности
кондор. Сверяемся с лимитами, заданными на холодную голову.

In [ ]:
LIMITS = {"delta": (-200, 200), "vega": (-50, 50)}
def check_limits(total, limits):
    for k, (lo, hi) in limits.items():
        val = getattr(total, k)
        print(f"чистая {k} {r(val):>8}  лимит [{lo}, {hi}]  ->", "ОК" if lo <= val <= hi else "ПРОБОЙ")
check_limits(total, LIMITS)

Здесь обе внутри лимитов. Если бы чистая дельта ушла за +200, вы бы захеджировали её вниз
(шорт акций / длинные путы, модуль 09) или подрезали бычью позицию, прежде чем добавлять новый риск.

## 3. Чек-лист в виде кода

Закодируйте чек-лист входа так, чтобы сделка, провалившая любое правило, отклонялась *до* отправки
заявки. Это дисциплина модуля 10, доведённая до механики.

In [ ]:
def pre_trade_check(pos, spot, vol, equity, risk_pct, iv_rank,
                    is_seller, event_in_horizon, book_total, limits):
    ml = analyzer.max_loss(pos)
    units, _ = size_position(pos, equity, risk_pct)
    proj = book_total + greeks.position_greeks(pos, spot, vol)
    checks = {
        "ограниченный риск и размер >= 1": ml != float("-inf") and units and units >= 1,
        "верная колонка IV": (iv_rank >= 50) == is_seller,
        "нет события в горизонте": not event_in_horizon,
        "лимит дельты после сделки": limits["delta"][0] <= proj.delta <= limits["delta"][1],
        "лимит веги после сделки": limits["vega"][0] <= proj.vega <= limits["vega"][1],
    }
    return checks, all(checks.values())

In [ ]:
# Кондор продавца при высокой IV, без событий, с корректным размером -> должен пройти
checks, ok = pre_trade_check(ic, SPOT, VOL, 25_000, 0.02, iv_rank=65,
                             is_seller=True, event_in_horizon=False,
                             book_total=greeks.Greeks(0,0,0,0,0), limits=LIMITS)
for k, v in checks.items():
    print(f"  {'ПРОЙДЕНО' if v else 'ПРОВАЛ'}  {k}")
print("ОТПРАВЛЯТЬ СДЕЛКУ" if ok else "НЕ ОТПРАВЛЯТЬ")

Теперь нарушим правило: тот же кондор (то есть *продавец* премии) при **низком IV rank
(12)** — вы продавали бы дешёвые опционы, находясь не на той стороне колонки волатильности. Чек-лист
обязан его отклонить.

In [ ]:
checks, ok = pre_trade_check(ic, SPOT, VOL, 25_000, 0.02, iv_rank=12,
                             is_seller=True, event_in_horizon=False,
                             book_total=greeks.Greeks(0,0,0,0,0), limits=LIMITS)
print({k: ("ПРОЙДЕНО" if v else "ПРОВАЛ") for k, v in checks.items()})
print("ОТПРАВЛЯТЬ СДЕЛКУ" if ok else "НЕ ОТПРАВЛЯТЬ")

## Эксперименты

1. Поменяйте правило размера на 1% и на 5% риска. Как меняется число кондоров на счёте в \$50k?
2. Добавьте четвёртую позицию (например, длинный календарь или длинный колл) и пересоберите
   агрегат. Удаётся ли вернуть чистую вегу к нулю позицией с длинной вегой в качестве балласта?
3. Ужмите `LIMITS["delta"]` до (-100, 100). Проходит ли текущая книга? Что бы вы подрезали?
4. Передайте проходившему кондору `event_in_horizon=True` — убедитесь, что чек-лист теперь его
   отклоняет.
5. Расширьте `pre_trade_check` булевым аргументом «ликвидность пройдена» и сделайте его
   обязательным. На реальных сделках чаще всего валятся именно здесь.